<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/demos/clf_cbf_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CBF-CLF-QP Filter Demo

This notebook demonstrates the use of a Control Barrier Function–Control Lyapunov Function Quadratic Program (CBF-CLF-QP) filter.

The implementation leverages two JAX-based Python packages:
- `cbfax`: Provides foundational tools for working with Control Barrier Functions (CBFs) and Control Lyapunov Functions (CLFs).
- `dynamaxsys`: Offers a collection of commonly used dynamical systems.

For a deeper understanding of their functionality, you can explore the source code here:
- `cbfax`: https://github.com/UW-CTRL/cbfax
- `dynamaxsys`: https://github.com/UW-CTRL/dynamaxsys


In [ ]:
!pip install dynamaxsys cbfax

In [ ]:
import functools
from typing import Callable, List, Tuple

import cvxpy as cp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from cbfax import (
    ControlBarrierFunction,
    ControlLyapunovFunction,
)
from cbfax.plotting import plot_cbf, plot_clf, plot_halfspace
from dynamaxsys import (
    ControlAffineDynamics,
    DynamicallyExtendedUnicycle,
    get_discrete_time_dynamics,
)
from ipywidgets import interact
from matplotlib.patches import Rectangle


Some helper functions for plotting

In [ ]:
def rotate_vector_ccw(vector: jnp.ndarray, theta: float) -> jnp.ndarray:
    """Rotate a vector counter-clockwise by an angle theta."""
    return (
        jnp.array([[jnp.cos(theta), -jnp.sin(theta)], [jnp.sin(theta), jnp.cos(theta)]])
        @ vector
    )

def plot_circle(
    ax: plt.Axes,
    center: jnp.ndarray,
    radius: float,
    color: str = "r",
    alpha: float = 0.3,
    linestyle: str = "-",
    linewidth: float = 2,
    fill: bool = False,
    **kwargs,
):
    """Plot a circle on the given axis."""
    circle = plt.Circle(
        (center[0], center[1]),
        radius,
        edgecolor=color,
        facecolor=color if fill else "none",
        alpha=alpha,
        linestyle=linestyle,
        linewidth=linewidth,
        fill=fill,
        **kwargs,
    )
    ax.add_patch(circle)
    return ax


def plot_car(
    ax: plt.Axes,
    state: jnp.ndarray,
    car_length: float = 0.5,
    car_width: float = 0.3,
    color: str = "lightskyblue",
    alpha: float = 0.6,
):
    """Plot a rectangle car on a given axis."""
    pos = state[:2]
    theta = state[2]
    left_corner = pos + rotate_vector_ccw(
        0.5 * jnp.array([-car_length, -car_width]), theta
    )
    car = Rectangle(
        left_corner,
        car_length,
        car_width,
        angle=theta * 180 / jnp.pi,
        color=color,
        alpha=alpha,
    )
    ax.add_patch(car)
    v_vector = jnp.stack(
        [pos, pos + 3 * car_length / 4 * jnp.array([np.cos(theta), np.sin(theta)])]
    )
    ax.plot(v_vector[:, 0], v_vector[:, 1])
    return ax

This class implements the CBF-CLF-QP filter. It takes the required inputs during initialization, constructs a `cvxpy` problem to enforce control Lyapunov and barrier function constraints, provides functions to update the problem parameters, and includes a method to solve for the filtered control input.

Take a look at what this class is doing.

In [ ]:
class CBFCLFFilter:
    clfs: List[ControlLyapunovFunction]
    cbfs: List[ControlBarrierFunction]
    clf_alphas: List[Callable[float, float]]
    cbf_alphas: List[Callable[float, float]]
    dynamics: ControlAffineDynamics
    control_limits: List[Tuple[float, float]]

    def __init__(
        self,
        clfs: List[ControlLyapunovFunction],
        cbfs: List[ControlBarrierFunction],
        clf_alphas: List[Callable[float, float]],
        cbf_alphas: List[Callable[float, float]],
        dynamics: ControlAffineDynamics,
        control_limits: List[Tuple[float, float]] = None,
        cbf_slack_weighting: List[float] = None,
        clf_slack_weighting: List[float] = None,
    ):
        self.clfs = clfs
        self.cbfs = cbfs
        self.clf_alphas = clf_alphas
        self.cbf_alphas = cbf_alphas
        self.dynamics = dynamics
        self.control_limits = control_limits

        # Assert that all CLFs have the same state_dim and all CBFs have the same state_dim, and that they match
        if self.clfs:
            clf_state_dim = self.clfs[0].state_dim
            assert all(clf.state_dim == clf_state_dim for clf in self.clfs), (
                "All CLFs must have the same state_dim"
            )
        if self.cbfs:
            cbf_state_dim = self.cbfs[0].state_dim
            assert all(cbf.state_dim == cbf_state_dim for cbf in self.cbfs), (
                "All CBFs must have the same state_dim"
            )
        if self.clfs and self.cbfs:
            assert clf_state_dim == cbf_state_dim, (
                "CLFs and CBFs must have the same state_dim"
            )

        assert self.dynamics.state_dim == clf_state_dim, (
            "Dynamics and CLFs must have the same state_dim"
        )

        # Obtain the control dimension, number of CLFs, and number of CBFs
        control_dim = self.dynamics.control_dim
        n_clf = len(self.clfs)
        n_cbf = len(self.cbfs)

        # Set up slack weighting if not provided
        if cbf_slack_weighting is None:
            cbf_slack_weighting = [1000] * len(self.cbfs)
        if clf_slack_weighting is None:
            clf_slack_weighting = [100] * len(self.clfs)


        # Set up CLF filter cvxpy problem
        # - Define the filtered control variable
        self.filtered_control = cp.Variable(control_dim, name="filtered_control")
        # - Define the linear and constant terms for each CLF
        self.linear_clf_terms = [
            cp.Parameter(control_dim, name=f"linear_clf_terms_{i}")
            for i in range(n_clf)
        ]
        self.constant_clf_terms = [
            cp.Parameter(name=f"constant_clf_terms_{i}") for i in range(n_clf)
        ]
        # - Define the slack variables for each CLF
        self.ep_clf = [cp.Variable(1, name=f"ep_clf_{i}") for i in range(n_clf)]
        # - Define the linear and constant terms for each CBF
        self.linear_cbf_terms = [
            cp.Parameter(control_dim, name=f"linear_cbf_terms_{i}")
            for i in range(n_cbf)
        ]
        self.constant_cbf_terms = [
            cp.Parameter(name=f"constant_cbf_terms_{i}") for i in range(n_cbf)
        ]
        # - Define the slack variables for each CBF
        self.ep_cbf = [cp.Variable(1, name=f"ep_cbf_{i}") for i in range(n_cbf)]

        # - Define the desired control input
        self.desired_control = cp.Parameter(control_dim)

        # - Define the slack weighting for each CLF and CBF
        self.clf_slack_weighting = [
            cp.Constant(clf_slack_weighting[i], name=f"clf_slack_weighting_{i}")
            for i in range(n_clf)
        ]
        self.cbf_slack_weighting = [
            cp.Constant(cbf_slack_weighting[i], name=f"cbf_slack_weighting_{i}")
            for i in range(n_cbf)
        ]

        # - Define the objective function
        objective = cp.Minimize(
            # - Minimize the control error
            cp.sum_squares(self.filtered_control - self.desired_control)
            # - Minimize the slack variables
            + sum(
                [
                    self.clf_slack_weighting[i] * cp.sum_squares(self.ep_clf[i])
                    for i in range(n_clf)
                ]
            )
            + sum(
                [
                    self.cbf_slack_weighting[i] * cp.sum_squares(self.ep_cbf[i])
                    for i in range(n_cbf)
                ]
            )
        )

        # - Define the constraints
        constraints = [
            # - CLF constraints
            self.linear_clf_terms[i] @ self.filtered_control
            + self.constant_clf_terms[i]
            <= self.ep_clf[i]
            for i in range(n_clf)
        ]
        constraints += [
            # - CBF constraints
            self.linear_cbf_terms[i] @ self.filtered_control
            + self.constant_cbf_terms[i]
            >= -self.ep_cbf[i]
            for i in range(n_cbf)
        ]
        # - Slack variables must be non-negative
        constraints += [self.ep_clf[i] >= 0 for i in range(n_clf)]
        constraints += [self.ep_cbf[i] >= 0 for i in range(n_cbf)]
        constraints += [
            # - Control limits
            self.filtered_control >= self.control_limits[0],
            self.filtered_control <= self.control_limits[1],
        ]

        # - Construct the cvxpy problem
        self.problem = cp.Problem(objective, constraints)

    def update_cvxpy_parameters(self, state: jnp.ndarray, desired_control: jnp.ndarray, time: float = 0.0):
        """Update the cvxpy problem parameters with the current state and desired control input."""
        # - Update the CLF parameters
        for i, (clf, alpha) in enumerate(zip(self.clfs, self.clf_alphas)):
            linear, constant = clf.control_constraint(state, time)
            self.linear_clf_terms[i].value = np.array(linear)
            self.constant_clf_terms[i].value = np.array(constant)

        # - Update the CBF parameters
        for i, (cbf, alpha) in enumerate(zip(self.cbfs, self.cbf_alphas)):
            linear, constant = cbf.control_constraint(state, time)
            self.linear_cbf_terms[i].value = np.array(linear)
            self.constant_cbf_terms[i].value = np.array(constant)

        # - Update the desired control input
        self.desired_control.value = np.array(desired_control)

    def get_constraint_values(self, state: jnp.ndarray, time: float = 0.0):
        """Get the constraint values for the current state and time."""
        constraint_terms = []
        signs = []
        n_clf = len(self.clfs)
        n_cbf = len(self.cbfs)
        # - Get the CLF constraint values
        for i, (clf, alpha) in enumerate(zip(self.clfs, self.clf_alphas)):
            linear, constant = clf.control_constraint(state, time)
            constraint_terms.append([linear, constant])
            signs.append("<=")

        # - Get the CBF constraint values
        for i, (cbf, alpha) in enumerate(zip(self.cbfs, self.cbf_alphas)):
            linear, constant = cbf.control_constraint(state, time)
            constraint_terms.append([linear, constant])
            signs.append(">=")
        return constraint_terms, signs, n_clf, n_cbf

    def solve(self, verbose=False):
        """Solve the cvxpy problem and return the filtered control input."""
        self.problem.solve(verbose=verbose)
        return self.filtered_control.value


Now, we can define our problem
We consider a dynamically-extended unicycle model. Its task is to reach a goal state, but there is an obstacle between the starting state and the goal state. Thus the system must move around the obstacle in order to reach the goal state.

The system also has control limits.

In [ ]:
dynamics_ct = DynamicallyExtendedUnicycle()

# Problem parameters
initial_state = jnp.array([-7.0, 0.1, 0.0, 2.0])
goal_state = jnp.array([10.0, 0.0, 0.0, 0.0])
obstacle_state = jnp.array([0.0, 0.0])
obstacle_radius = 1.5
# control: omega, acceleration
control_limits = [jnp.array([-jnp.pi / 3, -2.0]), jnp.array([jnp.pi / 3, 2.0])]
desired_control = jnp.array([0.0, 0.0])


# Simulation parameters
time_horizon = 10
dt = 0.01
eps = 1e-1
n_steps = int(time_horizon / dt)
dynamics_dt = get_discrete_time_dynamics(dynamics_ct, dt)

We can visualize the environment.

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
rest_values = initial_state[2:]
ax.scatter(
    initial_state[0], initial_state[1], color="blue", marker="o", zorder=10,  s=15, label="initial state"
)
ax.scatter(
    goal_state[0], goal_state[1], color="red", marker="x", zorder=10, label="goal state"
)
plot_circle(ax, obstacle_state, obstacle_radius, color="red", fill=True, label="obstacle")
plot_car(ax, car_length=2.0, car_width=1.0, state=initial_state, alpha=0.7)
ax.grid(alpha=0.3)
ax.legend()
ax.axis("equal")
ax.set_title("Environment")

Now, we set up the CBF and CLF parameters needed to construct our `CBFCLFFilter` class

In [ ]:
# Define the CLF functions
# - v1 is concerned with the distance to the goal
def v1_func(state, goal_state):
    return jnp.linalg.norm(state[:2] - goal_state[:2]) ** 2

# - v2 is concerned with the heading to the goal
def v2_func(state, goal_state):
    x, y, th = state[:3]
    x_g, y_g, th_g = goal_state[:3]
    return (th - jnp.atan2(y_g - y, x_g - x)) ** 2

# Define the CBF function
# - cbf is concerned with the distance to the obstacle
def cbf_func(state, obstacle_state, obstacle_radius):
    return 10 * (jnp.linalg.norm(state[:2] - obstacle_state) ** 2 - obstacle_radius**2)


# Define the alpha parameters for the CLF and CBF functions
clf_alpha = 0.5
cbf_alpha = 2.0
clf_alphas = [
    [lambda x: clf_alpha * x, lambda x: clf_alpha * x],
    lambda x: clf_alpha * x,
]
cbf_alphas = [[lambda x: cbf_alpha * x, lambda x: cbf_alpha  * x]]


# Use classes from `cbfax` to define the CLF and CBF functions
clf1 = ControlLyapunovFunction(
    functools.partial(v1_func, goal_state=goal_state),
    alpha_func=clf_alphas[0],
    dynamics=dynamics_ct,
    state_dim=dynamics_ct.state_dim,
    relative_degree=2
)
clf2 = ControlLyapunovFunction(
    functools.partial(v2_func, goal_state=goal_state),
    alpha_func=clf_alphas[1],
    dynamics=dynamics_ct,
    state_dim=dynamics_ct.state_dim,
    relative_degree=1
)
cbf = ControlBarrierFunction(
    functools.partial(
        cbf_func, obstacle_state=obstacle_state, obstacle_radius=obstacle_radius
    ),
    alpha_func=cbf_alphas[0],
    dynamics=dynamics_ct,
    state_dim=dynamics_ct.state_dim,
    relative_degree=2
)

# Put the CLF and CBF functions into lists
clfs = [clf1, clf2]
cbfs = [cbf]

# Define the slack weighting for the CLF and CBF functions
cbf_slack_weighting = [1000] * len(cbfs)
clf_slack_weighting = [100] * len(clfs)


In [ ]:
# Construct the `CBFCLFFilter` class
cbf_clf_filter = CBFCLFFilter(
    clfs=clfs,
    cbfs=cbfs,
    clf_alphas=clf_alphas,
    cbf_alphas=cbf_alphas,
    dynamics=dynamics_ct,
    control_limits=control_limits,
    cbf_slack_weighting=cbf_slack_weighting,
    clf_slack_weighting=clf_slack_weighting,
)

We are now ready to run the filter. Here, we assume the desired control input remains constant for simplicity.
However, in practice, the desired control may be provided by a trajectory planner such as MPC and can vary at each timestep.

Note that in our simulation, we step the system forward using *discrete-time* dynamics.

In [ ]:
# We will run the filter for `n_steps` timesteps and plot the results.

trajectory = [initial_state] # Store the trajectory
controls = [] # Store the controls
for step in range(n_steps):
    state = trajectory[-1] # Get the last state
    cbf_clf_filter.update_cvxpy_parameters(state, desired_control) # Update the filter parameters
    filtered_control = cbf_clf_filter.solve() # Solve the filter
    state = dynamics_dt(state, filtered_control) # Propagate the state
    trajectory.append(state) # Store the state
    controls.append(filtered_control) # Store the control
    if clf1(state) < eps: # Check if the goal has been reached
        print("Goal reached!")
        break
trajectory = jnp.stack(trajectory)
controls = jnp.stack(controls)


Let's plot the results!

In [ ]:
from cProfile import label
from IPython.display import clear_output

@interact(time_step=(0, len(trajectory) - 1))
def plot_interactive(time_step):
    clear_output(wait=True)
    fig, axs = plt.subplots(2, 3, figsize=(12, 6))
    ax = axs[0, 0]
    state = trajectory[time_step]
    control = controls[time_step]
    rest_values = state[2:]

    plt.subplot(2, 3, 1)

    plt.scatter(
        state[0], state[1], color="blue", marker="o", zorder=10, label="state", s=7
    )
    plt.scatter(
        goal_state[0], goal_state[1], color="red", marker="x", zorder=10, label="goal"
    )
    plt.plot(
        trajectory[:, 0], trajectory[:, 1], color="green", zorder=0, label="trajectory"
    )

    plot_cbf(cbf, rest_values=rest_values, fill=False, zorder=-10)
    plot_car(ax, car_length=2.0, car_width=1.0, state=state, alpha=0.7)
    plt.legend(ncol=3)
    plt.axis("equal")
    plt.title("Trajectory and CBF")

    plt.subplot(2, 3, 2)

    plt.plot(controls[:, 0], color="blue", zorder=0, label="steering")
    plt.plot(controls[:, 1], color="red", zorder=0, label="acceleration")
    plt.scatter(time_step, control[0], color="blue", marker="o", zorder=10, s=5)
    plt.scatter(time_step, control[1], color="red", marker="o", zorder=10, s=5)
    plt.grid(alpha=0.3)
    plt.xlabel("Time step")
    plt.ylabel("Control")
    plt.legend(fontsize=8)
    plt.title("Control trajectory")

    plt.subplot(2, 3, 3)
    constraint_terms, signs, n_clf, n_cbf = cbf_clf_filter.get_constraint_values(state)
    colors = plt.cm.tab10.colors  # 10 distinct colors
    for (i, ((linear, constant), sign)) in enumerate(zip(constraint_terms, signs)):

        name = f"CLF {i+1}" if i < n_clf else f"CBF {i - n_clf + 1}"
        plot_halfspace(
            linear,
            constant,
            sign,
            xlim=(control_limits[0][0], control_limits[1][0]),
            ylim=(control_limits[0][1], control_limits[1][1]),
            color=colors[i],
            name=name
        )
    plt.scatter(
        control[0], control[1], color="blue", marker="o", zorder=10, label="control"
    )
    plt.scatter(desired_control[0], desired_control[1], color="red", marker="x", zorder=10, label="desired control")
    plt.legend(ncol=3, fontsize=8)
    plt.xlim(control_limits[0][0], control_limits[1][0])
    plt.ylim(control_limits[0][1], control_limits[1][1])
    # plt.axis("equal")
    plt.title("Control constraints")

    plt.subplot(2, 3, 4)

    plot_cbf(cbf, rest_values=rest_values)
    plt.scatter(goal_state[0], goal_state[1], color="red", marker="x", zorder=10)
    plt.scatter(state[0], state[1], color="blue", marker="o", zorder=10, label="state")
    plt.title("CBF")


    plt.subplot(2, 3, 5)
    plot_clf(clf1, rest_values=rest_values)
    plt.scatter(goal_state[0], goal_state[1], color="red", marker="x", zorder=10)
    plt.scatter(state[0], state[1], color="blue", marker="o", zorder=10, label="state")
    plt.title("CLF 1 (position reaching)")

    plt.subplot(2, 3, 6)
    plot_clf(clf2, rest_values=rest_values)
    plt.scatter(goal_state[0], goal_state[1], color="red", marker="x", zorder=10)
    plt.scatter(state[0], state[1], color="blue", marker="o", zorder=10, label="state")
    plt.title("CLF 2 (heading reaching)")
    plt.tight_layout()



We can also plot how the control lyapunov and control barrier functions changes over the course of the trajectory.

In [ ]:
clf1_values = jax.vmap(clf1)(trajectory)
clf2_values = jax.vmap(clf2)(trajectory)
cbf_values = jax.vmap(cbf)(trajectory)

plt.plot(clf1_values, label="CLF 1")
plt.plot(clf2_values, label="CLF 2")
plt.plot(cbf_values, label="CBF")
plt.legend()
plt.xlabel("Time step")
plt.ylabel("Value")
plt.title("CLF and CBF values")
plt.grid(alpha=0.3)
